# Biomedical RAG over PubMed — Alzheimer's Disease Therapeutic Targets

A retrieval-augmented generation (RAG) pipeline that answers questions about **Alzheimer's disease drug targets** grounded in real PubMed literature.

**Pipeline:** PubMed collection (Entrez) → text cleaning → chunking → dense embeddings (PubMedBERT) + ChromaDB → hybrid retrieval (dense + BM25) → local LLM generation (Ollama) → answer validation & evaluation → interactive UI.

> Set your NCBI contact email before running the collection step: `export ENTREZ_EMAIL="you@example.com"`.
> Generation requires a local [Ollama](https://ollama.com) server with a small model (e.g. `ollama pull llama3.2:3b`).


In [1]:
# Imports
import json
import time
from pathlib import Path
from Bio import Entrez
from tqdm.notebook import tqdm

In [2]:
# Config
import os

# PubMed (NCBI Entrez) requires a contact email. Set it via env var:
#   export ENTREZ_EMAIL="your.email@example.com"
EMAIL = os.environ.get("ENTREZ_EMAIL", "your.email@example.com")
API_KEY = os.environ.get("NCBI_API_KEY")  # optional, raises rate limit to 10 req/s
TARGET_COUNT = 100
YEARS = (2010, 2026)


In [3]:
# Setup Entrez
Entrez.email = EMAIL
if API_KEY:
    Entrez.api_key = API_KEY
    DELAY = 0.1  # 10 req/sec
else:
    DELAY = 0.34  # 3 req/sec

In [4]:
# Query sets для разнообразия
QUERIES = [
    # Широкие
    "Alzheimer's disease therapeutic targets",
    "Alzheimer disease drug targets",
    
    # Белковые таргеты
    "Alzheimer amyloid beta target",
    "Alzheimer tau protein target",
    "BACE1 inhibitor Alzheimer",
    "gamma secretase Alzheimer",
    
    # Воспаление
    "neuroinflammation Alzheimer target",
    "microglia Alzheimer therapy",
    "TREM2 Alzheimer",
    
    # Синапсы
    "synaptic dysfunction Alzheimer",
    "NMDA receptor Alzheimer",
    
    # Клинические
    "Alzheimer clinical trial drug target",
]

# Добавляем фильтр по годам
QUERIES = [f"{q} AND {YEARS[0]}:{YEARS[1]}[pdat]" for q in QUERIES]

In [5]:
def search_pubmed(query, max_results=20):
    """Поиск статей"""
    try:
        handle = Entrez.esearch(
            db="pubmed",
            term=query,
            retmax=max_results,
            sort="relevance"
        )
        record = Entrez.read(handle)
        handle.close()
        time.sleep(DELAY)
        return record["IdList"]
    except Exception as e:
        print(f"Error: {e}")
        return []

In [6]:
def fetch_paper(pmid):
    """Получить метаданные статьи"""
    try:
        handle = Entrez.efetch(db="pubmed", id=pmid, rettype="xml")
        records = Entrez.read(handle)
        handle.close()
        time.sleep(DELAY)
        
        article = records['PubmedArticle'][0]
        medline = article['MedlineCitation']
        art = medline['Article']
        
        # Title
        title = str(art.get('ArticleTitle', ''))
        
        # Abstract
        abs_parts = art.get('Abstract', {}).get('AbstractText', [])
        if isinstance(abs_parts, list):
            abstract = ' '.join([str(p) for p in abs_parts])
        else:
            abstract = str(abs_parts)
        
        # Пропускаем если нет нормального абстракта
        if not abstract or len(abstract) < 50:
            return None
        
        # Authors
        authors = []
        for a in art.get('AuthorList', [])[:10]:
            if 'LastName' in a and 'Initials' in a:
                authors.append(f"{a['LastName']} {a['Initials']}")
        
        # Journal & Date
        journal = art.get('Journal', {}).get('Title', 'Unknown')
        pub_date = art.get('Journal', {}).get('JournalIssue', {}).get('PubDate', {})
        year = pub_date.get('Year', '')
        
        # DOI
        doi = None
        for id_item in article.get('PubmedData', {}).get('ArticleIdList', []):
            if id_item.attributes.get('IdType') == 'doi':
                doi = str(id_item)
        
        # MeSH terms
        mesh = [str(m.get('DescriptorName', '')) 
                for m in medline.get('MeshHeadingList', [])[:10]]
        
        return {
            'id': pmid,
            'title': title,
            'abstract': abstract,
            'authors': authors,
            'journal': journal,
            'year': year,
            'doi': doi,
            'mesh_terms': mesh,
            'source': 'pubmed'
        }
        
    except Exception as e:
        print(f"Failed PMID {pmid}: {e}")
        return None

In [7]:
# Сбор данных
papers = []
seen_pmids = set()

with tqdm(total=TARGET_COUNT, desc="Collecting") as pbar:
    for query in QUERIES:
        if len(papers) >= TARGET_COUNT:
            break
        
        print(f"\nQuery: {query[:60]}...")
        pmids = search_pubmed(query, max_results=20)
        new_pmids = [p for p in pmids if p not in seen_pmids]
        print(f"Found {len(pmids)} total, {len(new_pmids)} new")
        
        for pmid in new_pmids:
            if len(papers) >= TARGET_COUNT:
                break
            
            paper = fetch_paper(pmid)
            if paper:
                papers.append(paper)
                seen_pmids.add(pmid)
                pbar.update(1)

print(f"\nOK Collected {len(papers)} papers")

Collecting:   0%|          | 0/100 [00:00<?, ?it/s]


Query: Alzheimer's disease therapeutic targets AND 2010:2026[pdat]...
Found 20 total, 20 new

Query: Alzheimer disease drug targets AND 2010:2026[pdat]...
Found 20 total, 16 new

Query: Alzheimer amyloid beta target AND 2010:2026[pdat]...
Found 20 total, 15 new

Query: Alzheimer tau protein target AND 2010:2026[pdat]...
Found 20 total, 13 new

Query: BACE1 inhibitor Alzheimer AND 2010:2026[pdat]...
Found 20 total, 19 new

Query: gamma secretase Alzheimer AND 2010:2026[pdat]...
Found 20 total, 19 new

Query: neuroinflammation Alzheimer target AND 2010:2026[pdat]...
Found 20 total, 18 new

OK Collected 100 papers


In [8]:
# Сохранение
output_dir = Path("data/articles")
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "papers.jsonl"

with open(output_file, 'w', encoding='utf-8') as f:
    for paper in papers:
        f.write(json.dumps(paper, ensure_ascii=False) + '\n')

print(f"OK Saved to {output_file}")

OK Saved to data/articles/papers.jsonl


In [9]:
# Быстрая статистика
import pandas as pd
from collections import Counter

df = pd.DataFrame(papers)
print(f"Total papers: {len(df)}")
print(f"\nYears:")
print(df['year'].value_counts().sort_index())
print(f"\nTop journals:")
print(df['journal'].value_counts().head(5))
print(f"\nAbstract lengths:")
df['abs_len'] = df['abstract'].str.len()
print(df['abs_len'].describe())

Total papers: 100

Years:
year
2012     3
2013     2
2015     1
2016     4
2017     2
2018     2
2019     7
2020    14
2021    12
2022    10
2023    11
2024    17
2025    15
Name: count, dtype: int64

Top journals:
journal
Molecules (Basel, Switzerland)                 6
International journal of molecular sciences    4
Brain : a journal of neurology                 4
Journal of Alzheimer's disease : JAD           4
Neuron                                         3
Name: count, dtype: int64

Abstract lengths:
count     100.000000
mean     1406.490000
std       392.889953
min       103.000000
25%      1163.250000
50%      1370.000000
75%      1638.000000
max      2675.000000
Name: abs_len, dtype: float64


In [10]:
# Пример статьи
print("Example paper:")
print(json.dumps(papers[0], indent=2, ensure_ascii=False))

Example paper:
{
  "id": "36430432",
  "title": "Alzheimer's Disease: Treatment Strategies and Their Limitations.",
  "abstract": "Alzheimer's disease (AD) is the most frequent case of neurodegenerative disease and is becoming a major public health problem all over the world. Many therapeutic strategies have been explored for several decades; however, there is still no curative treatment, and the priority remains prevention. In this review, we present an update on the clinical and physiological phase of the AD spectrum, modifiable and non-modifiable risk factors for AD treatment with a focus on prevention strategies, then research models used in AD, followed by a discussion of treatment limitations. The prevention methods can significantly slow AD evolution and are currently the best strategy possible before the advanced stages of the disease. Indeed, current drug treatments have only symptomatic effects, and disease-modifying treatments are not yet available. Drug delivery to the cent

In [11]:
# Load data
papers = []
with open('data/articles/papers.jsonl') as f:
    for line in f:
        papers.append(json.loads(line))

print(f"Loaded {len(papers)} papers")

Loaded 100 papers


In [12]:
# DataFrame
df = pd.DataFrame(papers)
df['abs_len'] = df['abstract'].str.len()
df['n_authors'] = df['authors'].apply(len)
df.head()

,id,title,abstract,authors,journal,year,doi,mesh_terms,source,abs_len,n_authors
0,36430432,Alzheimer's Disease: Treatment Strategies and ...,Alzheimer's disease (AD) is the most frequent ...,"[Passeri E, Elkhoury K, Morsink M, Broersen K,...",International journal of molecular sciences,2022,10.3390/ijms232213954,"[Humans, Alzheimer Disease, Neurodegenerative ...",pubmed,1789,9
1,34181171,Recent advances on drug development and emergi...,Alzheimer's disease (AD) is a neurodegenerativ...,"[Athar T, Al Balushi K, Khan SA]",Molecular biology reports,2021,10.1007/s11033-021-06512-9,"[Alzheimer Disease, Animals, Biomarkers, Clini...",pubmed,1375,3
2,34813026,Recent advances in molecular pathways and ther...,Alzheimer's disease (AD) is a major contributo...,"[Dhapola R, Hota SS, Sarma P, Bhattacharyya A,...",Inflammopharmacology,2021,10.1007/s10787-021-00889-6,"[Alzheimer Disease, Animals, Cytokines, Humans...",pubmed,1605,6
3,38513667,P-tau217 correlates with neurodegeneration in ...,Neuronal loss is the central issue in Alzheime...,"[Zhang D, Zhang W, Ming C, Gao X, Yuan H, Lin ...",Neuron,2024,10.1016/j.neuron.2024.02.017,"[Aged, Aged, 80 and over, Animals, Female, Hum...",pubmed,1154,10
4,33652356,Alzheimer's disease and its treatment by diffe...,Alzheimer's disease (AD) is a neurodegenerativ...,"[Srivastava S, Ahmad R, Khare SK]",European journal of medicinal chemistry,2021,10.1016/j.ejmech.2021.113320,"[Alzheimer Disease, Animals, Biomarkers, Blood...",pubmed,1479,3


In [13]:
# MeSH terms analysis
all_mesh = []
for paper in papers:
    if paper.get('mesh_terms'):
        all_mesh.extend(paper['mesh_terms'])

mesh_counts = Counter(all_mesh)
print("\nTop 20 MeSH terms:")
for term, count in mesh_counts.most_common(20):
    print(f"{count:3d} | {term}")


Top 20 MeSH terms:
 99 | Alzheimer Disease
 92 | Humans
 60 | Animals
 55 | Amyloid beta-Peptides
 38 | Amyloid Precursor Protein Secretases
 20 | tau Proteins
 18 | Drug Delivery Systems
 18 | Aspartic Acid Endopeptidases
 15 | Brain
 15 | Amyloid beta-Protein Precursor
 13 | Mice
 12 | Blood-Brain Barrier
 11 | Aged
 10 | Disease Models, Animal
  9 | Immunotherapy
  8 | Biomarkers
  8 | Male
  7 | Nanoparticles
  7 | Clinical Trials as Topic
  7 | Female


In [14]:
# Sample papers
print("Random sample:")
sample = df.sample(3)
for idx, row in sample.iterrows():
    print(f"\n{'='*60}")
    print(f"PMID: {row['id']}")
    print(f"Title: {row['title']}")
    print(f"Journal: {row['journal']} ({row['year']})")
    print(f"Authors: {', '.join(row['authors'][:5])}...")
    print(f"Abstract: {row['abstract'][:200]}...")


Random sample:

PMID: 35883556
Title: Comparison of Tau and Amyloid-β Targeted Immunotherapy Nanoparticles for Alzheimer's Disease.
Journal: Biomolecules (2022)
Authors: Mashal Y, Abdelhady H, Iyer AK...
Abstract: Alzheimer's disease (AD) is a rapidly growing global concern associated with the accumulation of amyloid-β plaques and intracellular neurofibrillary tangles in the brain combined with a high acetylcho...

PMID: 40795314
Title: Restoring amyloid-β42 and γ-secretase function in Alzheimer's disease.
Journal: Brain : a journal of neurology (2025)
Authors: Espay AJ, Ezzat K, Kepp KP, Daly T, Robakis NK...
Abstract: Emerging evidence is challenging the long-standing notion that Alzheimer's disease (AD) is caused by increased γ-secretase function and overproduction of 42-amino acid amyloid-beta (Aβ42). CSF levels ...

PMID: 39755304
Title: Identification of potential therapeutic targets for Alzheimer's disease from the proteomes of plasma and cerebrospinal fluid in a multicenter Men

In [15]:
import re
import unicodedata
def clean_text(text):
    """Очистка текста"""
    # Убираем HTML/XML теги если есть
    text = re.sub(r'<[^>]+>', '', text)
    
    # Нормализуем пробелы
    text = re.sub(r'\s+', ' ', text)
    
    # Unicode NFC нормализация
    text = unicodedata.normalize('NFC', text)
    
    # Убираем лишние пробелы по краям
    text = text.strip()
    
    return text

In [16]:
# Test cleaning
test_text = "  This  is   a test  with   HTML <b>tags</b>  "
print(f"Before: '{test_text}'")
print(f"After:  '{clean_text(test_text)}'")

Before: '  This  is   a test  with   HTML <b>tags</b>  '
After:  'This is a test with HTML tags'


In [17]:
# Config
MIN_TEXT_LENGTH = 300  # минимальная длина текста

processed_docs = []
skipped = 0

for paper in papers:
    # Формируем текст: title + abstract
    title = clean_text(paper['title'])
    abstract = clean_text(paper['abstract'])
    
    text = f"{title}\n\n{abstract}"
    
    # Проверка длины
    if len(text) < MIN_TEXT_LENGTH:
        skipped += 1
        continue
    
    # Формируем документ
    doc = {
        'doc_id': f"pubmed:{paper['id']}",
        'title': title,
        'text': text,
        'year': paper.get('year', ''),
        'doi': paper.get('doi', ''),
        'url': f"https://pubmed.ncbi.nlm.nih.gov/{paper['id']}/",
        'source': 'pubmed',
        'authors': paper.get('authors', []),
        'journal': paper.get('journal', ''),
        'mesh_terms': paper.get('mesh_terms', [])
    }
    
    processed_docs.append(doc)

print(f"Processed {len(processed_docs)} documents")
print(f"Skipped {skipped} (too short)")
print(f"Pass rate: {len(processed_docs)/len(papers)*100:.1f}%")

Processed 99 documents
Skipped 1 (too short)
Pass rate: 99.0%


In [18]:
# Check for duplicates
doc_ids = [d['doc_id'] for d in processed_docs]
assert len(doc_ids) == len(set(doc_ids)), "Duplicate doc_ids found!"
print("OK All doc_ids unique")

OK All doc_ids unique


In [19]:
# Example doc
print(json.dumps(processed_docs[0], indent=2, ensure_ascii=False)[:500] + '...')

{
  "doc_id": "pubmed:36430432",
  "title": "Alzheimer's Disease: Treatment Strategies and Their Limitations.",
  "text": "Alzheimer's Disease: Treatment Strategies and Their Limitations.\n\nAlzheimer's disease (AD) is the most frequent case of neurodegenerative disease and is becoming a major public health problem all over the world. Many therapeutic strategies have been explored for several decades; however, there is still no curative treatment, and the priority remains prevention. In this rev...


In [20]:
# Save
PROJECT_DIR = Path("data")
processed_dir = PROJECT_DIR / "articles" / "processed_pubmed"
processed_dir.mkdir(parents=True, exist_ok=True) # Создаём путь, если его нет
docs_file = processed_dir / 'docs.jsonl'
with open(docs_file, 'w', encoding='utf-8') as f:
    for doc in processed_docs:
        f.write(json.dumps(doc, ensure_ascii=False) + '\n')

print(f"OK Saved {len(processed_docs)} docs to {docs_file}")

OK Saved 99 docs to data/articles/processed_pubmed/docs.jsonl


In [21]:
# Config
CHUNK_SIZE = 500  # символов
CHUNK_OVERLAP = 100  # символов

In [ ]:


from tqdm.notebook import tqdm

def split_into_chunks(text, chunk_size=800, overlap=120):
    """Разбить на чанки с overlap (с защитой от зацикливания)"""
    if not text or len(text) < 50:
        return []
    
    chunks = []
    start = 0
    text_len = len(text)
    max_iterations = text_len // (chunk_size - overlap) + 10  # Защита от бесконечного цикла
    iteration = 0
    
    while start < text_len and iteration < max_iterations:
        iteration += 1
        
        # Конец чанка
        end = min(start + chunk_size, text_len)
        
        # Пытаемся найти конец предложения
        if end < text_len:
            last_period = text[start:end].rfind('. ')
            if last_period > chunk_size // 2:  # Только если точка не в самом начале
                end = start + last_period + 2
        
        # Берем чанк
        chunk_text = text[start:end].strip()
        
        if chunk_text:
            chunks.append({
                'text': chunk_text,
                'start': start,
                'end': end
            })
        
        # ВАЖНО: гарантируем что start продвигается вперед
        next_start = end - overlap
        if next_start <= start:  # Защита от зацикливания
            next_start = start + max(1, chunk_size // 2)
        
        start = next_start
        
        if start >= text_len:
            break
    
    return chunks

# Обработка с прогресс-баром и обработкой ошибок

docs = []
all_chunks = []
errors = []

for paper in tqdm(papers, desc="Processing papers"):
    try:
        # Очищаем текст
        title = clean_text(paper['title'])
        abstract = clean_text(paper['abstract'])
        full_text = f"{title}\n\n{abstract}"
        
        # Проверка длины
        if len(full_text) < 100:
            print(f"Skipping short text: {paper['id']}")
            continue
        
        # Документ
        doc = {
            'doc_id': f"pubmed:{paper['id']}",
            'title': title,
            'text': full_text,
            'year': paper['year'],
            'url': f"https://pubmed.ncbi.nlm.nih.gov/{paper['id']}/",
            'authors': paper['authors'][:3],
            'journal': paper['journal']
        }
        docs.append(doc)
        
        # Чанки
        chunks = split_into_chunks(full_text, chunk_size=500, overlap=100)
        
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                'chunk_id': f"{doc['doc_id']}#c{i}",
                'doc_id': doc['doc_id'],
                'text': chunk['text'],
                'title': title,
                'url': doc['url'],
                'metadata': {
                    'year': paper['year'],
                    'journal': paper['journal'],
                    'authors': paper['authors'][:2]
                }
            })
    
    except Exception as e:
        errors.append({'pmid': paper['id'], 'error': str(e)})
        print(f"Error processing {paper['id']}: {e}")

print(f"\n{'='*60}")
print(f"Документов: {len(docs)}")
print(f"Чанков: {len(all_chunks)}")
print(f"Среднее чанков на документ: {len(all_chunks)/len(docs):.1f}")
if errors:
    print(f"Ошибок: {len(errors)}")
print(f"{'='*60}")

Processing papers:   0%|          | 0/100 [00:00<?, ?it/s]


Документов: 100
Чанков: 564
Среднее чанков на документ: 5.6


In [ ]:
# Test chunking (checks the active split_into_chunks implementation)
test_text = "First sentence. " * 50
test_chunks = split_into_chunks(test_text, chunk_size=100, overlap=20)
print(f"Test: {len(test_text)} chars -> {len(test_chunks)} chunks")
print(f"Chunk lengths: {[len(c['text']) for c in test_chunks[:3]]}")

In [25]:
# Chunk statistics
chunk_lengths = [len(c['text']) for c in all_chunks]
print(f"\nChunk length stats:")
print(f"  Min:  {min(chunk_lengths)}")
print(f"  Max:  {max(chunk_lengths)}")
print(f"  Mean: {sum(chunk_lengths)/len(chunk_lengths):.0f}")
print(f"  Median: {sorted(chunk_lengths)[len(chunk_lengths)//2]}")


Chunk length stats:
  Min:  99
  Max:  500
  Mean: 346
  Median: 384


In [26]:
# Example chunks
print("\nExample chunks from first doc:")
first_doc_chunks = [c for c in all_chunks if c['doc_id'] == processed_docs[0]['doc_id']]
for i, chunk in enumerate(first_doc_chunks[:3]):
    print(f"\n--- Chunk {i} (len={len(chunk['text'])}) ---")
    print(chunk['text'][:200] + '...')


Example chunks from first doc:

--- Chunk 0 (len=363) ---
Alzheimer's Disease: Treatment Strategies and Their Limitations.

Alzheimer's disease (AD) is the most frequent case of neurodegenerative disease and is becoming a major public health problem all over...

--- Chunk 1 (len=379) ---
everal decades; however, there is still no curative treatment, and the priority remains prevention. In this review, we present an update on the clinical and physiological phase of the AD spectrum, mod...

--- Chunk 2 (len=365) ---
ion strategies, then research models used in AD, followed by a discussion of treatment limitations. The prevention methods can significantly slow AD evolution and are currently the best strategy possi...


In [27]:
# Простой анализ упоминаний ключевых терминов
key_terms = {
    'amyloid': r'\bamyloid[-\s]beta\b|\bamyloid\b',
    'tau': r'\btau\s+protein\b|\btau\b',
    'BACE1': r'\bBACE1?\b',
    'APOE': r'\bAPOE\b',
    'neuroinflammation': r'\bneuroinflammation\b',
    'microglia': r'\bmicroglia\b',
    'TREM2': r'\bTREM2\b',
}

term_counts = {term: 0 for term in key_terms}
for doc in docs:
    for term, pattern in key_terms.items():
        if re.search(pattern, doc['text'], re.IGNORECASE):
            term_counts[term] += 1

print("\nКлючевые термины (количество документов):")
for term, count in sorted(term_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {term:20s}: {count:3d} ({count/len(docs)*100:.1f}%)")



Ключевые термины (количество документов):
  amyloid             :  75 (75.0%)
  tau                 :  41 (41.0%)
  BACE1               :  21 (21.0%)
  neuroinflammation   :  13 (13.0%)
  microglia           :   7 (7.0%)
  APOE                :   1 (1.0%)
  TREM2               :   1 (1.0%)


In [28]:
# Save chunks
chunks_file = processed_dir / 'chunks.jsonl'
with open(chunks_file, 'w', encoding='utf-8') as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f"OK Saved {len(all_chunks)} chunks to {chunks_file}")

OK Saved 564 chunks to data/articles/processed_pubmed/chunks.jsonl


In [ ]:
#!pip install sentence-transformers chromadb ollama ipywidgets

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions

print("Загружаю embedding модель...")
embedder = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO')
print("OK Модель загружена")

Загружаю embedding модель...
OK Модель загружена


In [30]:
# Инициализация ChromaDB
chroma_client = chromadb.Client()

# Удаляем коллекцию если существует
try:
    chroma_client.delete_collection("alzheimer_papers")
except:
    pass

# Создаём коллекцию
# ВАЖНО: задаём пространство HNSW как cosine, чтобы distance был (1 - cosine_similarity)
collection = chroma_client.create_collection(
    name="alzheimer_papers",
    metadata={
        "description": "Alzheimer's disease research papers",
        "hnsw:space": "cosine",
    },
)

print("OK Коллекция создана")
print("Space:", collection.metadata.get("hnsw:space", "(unknown)"))


OK Коллекция создана
Space: cosine


In [31]:
print(f"Добавляю {len(all_chunks)} чанков в векторную БД...")

# Batch добавление (по 100 чанков)
batch_size = 100
for i in range(0, len(all_chunks), batch_size):
    batch = all_chunks[i:i+batch_size]

    # Эмбеддинги (нормализуем, чтобы cosine работал корректно)
    texts = [c['text'] for c in batch]
    embeddings = embedder.encode(texts, normalize_embeddings=True).tolist()

    # Метаданные
    metadatas = []
    for c in batch:
        meta = {
            'title': c['title'],
            'url': c['url'],
            'year': str(c['metadata']['year']),
            'journal': c['metadata']['journal'][:100],  # Обрезаем длинные названия
        }
        metadatas.append(meta)

    # Добавляем в ChromaDB
    collection.add(
        ids=[c['chunk_id'] for c in batch],
        embeddings=embeddings,
        documents=texts,
        metadatas=metadatas
    )

    if (i + batch_size) % 200 == 0:
        print(f"  Добавлено {min(i+batch_size, len(all_chunks))}/{len(all_chunks)}")

print("OK Все чанки добавлены")


Добавляю 564 чанков в векторную БД...
  Добавлено 200/564
  Добавлено 400/564
  Добавлено 564/564
OK Все чанки добавлены


In [ ]:
"""
## Retrieval: Поиск релевантных чанков
"""

import re
import math
from collections import Counter
from typing import List, Dict, Any, Optional, Tuple


def _default_query_expansions(question: str):
    """Небольшой query expansion, чтобы вытягивать конкретные мишени."""
    targets = [
        "BACE1",
        "gamma secretase",
        "PSEN1",
        "PSEN2",
        "APP",
        "TREM2",
        "NLRP3",
        "APOE4",
        "APOE",
        "tau",
        "amyloid beta",
    ]
    return [f"{question} targets {' '.join(targets)}"]


def _bm25_tokenize(text: str) -> List[str]:
    """Простая токенизация для BM25 (регистронезависимая)."""
    if not text:
        return []
    text = text.replace("-", " ").lower()
    return re.findall(r"\b[a-z0-9]+\b", text)


class BM25Index:
    """Минималистичная реализация BM25 без внешних библиотек."""

    def __init__(self, documents: List[str], *, k1: float = 1.5, b: float = 0.75):
        self.k1 = float(k1)
        self.b = float(b)
        self.N = len(documents)
        self.docs_tokens = [_bm25_tokenize(d) for d in documents]
        self.doc_lens = [len(toks) for toks in self.docs_tokens]
        self.avgdl = (sum(self.doc_lens) / self.N) if self.N else 0.0
        self.tfs: List[Counter] = [Counter(toks) for toks in self.docs_tokens]
        self.dfs: Counter = Counter()
        for tf in self.tfs:
            for term in tf.keys():
                self.dfs[term] += 1

    def idf(self, term: str) -> float:
        df = self.dfs.get(term, 0)
        # классический BM25 idf, сглаженный
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1.0) if self.N else 0.0

    def score(self, query: str) -> List[float]:
        q_terms = _bm25_tokenize(query)
        if not q_terms or not self.N:
            return [0.0] * self.N
        scores = [0.0] * self.N
        for term in q_terms:
            idf = self.idf(term)
            if idf <= 0:
                continue
            for i in range(self.N):
                tf = self.tfs[i].get(term, 0)
                if tf == 0:
                    continue
                dl = self.doc_lens[i]
                denom = tf + self.k1 * (1.0 - self.b + self.b * (dl / self.avgdl if self.avgdl else 0.0))
                scores[i] += idf * (tf * (self.k1 + 1.0)) / (denom if denom else 1.0)
        return scores


def _minmax_norm(values: List[float]) -> List[float]:
    if not values:
        return []
    v_min = min(values)
    v_max = max(values)
    denom = v_max - v_min
    if denom <= 1e-12:
        return [1.0 for _ in values]
    return [(v - v_min) / denom for v in values]


# Build BM25 index once (based on all_chunks)
_BM25_DOCS = [c.get('text', '') for c in all_chunks]
_BM25_IDS = [c.get('chunk_id') for c in all_chunks]
_BM25_META_BY_ID = {
    c.get('chunk_id'): {
        'title': c.get('title', ''),
        'url': c.get('url', ''),
        'year': str(c.get('metadata', {}).get('year', '')),
        'journal': (c.get('metadata', {}).get('journal', '') or '')[:100],
    }
    for c in all_chunks
}
bm25_index = BM25Index(_BM25_DOCS)
print(f"BM25 index ready: {len(_BM25_DOCS)} documents")


def _dense_retrieve(query: str, n_results: int = 5, extra_queries: Optional[List[str]] = None) -> List[Dict[str, Any]]:
    """Dense retrieval через Chroma + SentenceTransformer (как раньше)."""
    queries = [query]
    if extra_queries:
        queries.extend([q for q in extra_queries if q and q.strip()])

    best_by_id: Dict[str, Dict[str, Any]] = {}
    for q in queries:
        q_emb = embedder.encode([q], normalize_embeddings=True).tolist()
        results = collection.query(query_embeddings=q_emb, n_results=n_results)

        ids = results.get('ids', [[]])[0]
        docs = results.get('documents', [[]])[0]
        dists = results.get('distances', [[]])[0]
        metas = results.get('metadatas', [[]])[0]

        for i in range(len(ids)):
            chunk_id = ids[i]
            dist = float(dists[i]) if dists and i < len(dists) else None
            item = {
                'chunk_id': chunk_id,
                'text': docs[i],
                'distance': dist,
                'bm25_score': None,
                'hybrid_score': None,
                'metadata': metas[i] if metas and i < len(metas) else {},
            }
            if chunk_id not in best_by_id:
                best_by_id[chunk_id] = item
            else:
                prev = best_by_id[chunk_id]
                if prev.get('distance') is None or (dist is not None and dist < prev.get('distance')):
                    best_by_id[chunk_id] = item

    merged = list(best_by_id.values())
    merged.sort(key=lambda x: (x['distance'] is None, x['distance']))
    return merged[:n_results]


def _bm25_retrieve(query: str, n_results: int = 5) -> List[Dict[str, Any]]:
    """Лексический retrieval через BM25 (быстро ловит редкие токены)."""
    scores = bm25_index.score(query)
    if not scores:
        return []
    # top-k по score
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n_results]
    out: List[Dict[str, Any]] = []
    for i in top_idx:
        chunk_id = _BM25_IDS[i]
        text = _BM25_DOCS[i]
        out.append({
            'chunk_id': chunk_id,
            'text': text,
            'distance': None,
            'bm25_score': float(scores[i]),
            'hybrid_score': None,
            'metadata': _BM25_META_BY_ID.get(chunk_id, {}),
        })
    return out


def retrieve_chunks(query: str, n_results: int = 5, extra_queries: Optional[List[str]] = None, *,
                    use_bm25: bool = True, bm25_k: int = 30, dense_k: int = 30, alpha: float = 0.65) -> List[Dict[str, Any]]:
    """Hybrid retrieval: dense (Chroma) + BM25, затем merge и ранжирование.

    Важно:
    - Для Chroma с hnsw:space="cosine": distance = 1 - cosine_similarity.
    - Query expansion применяется ТОЛЬКО к dense-части, чтобы BM25 не "перекошивать" по списку таргетов.
    """
    dense_candidates = _dense_retrieve(query, n_results=dense_k, extra_queries=extra_queries)
    bm25_candidates = _bm25_retrieve(query, n_results=bm25_k) if use_bm25 else []

    by_id: Dict[str, Dict[str, Any]] = {}
    for item in dense_candidates + bm25_candidates:
        cid = item['chunk_id']
        if cid not in by_id:
            by_id[cid] = item
        else:
            # оставим лучший distance, но не потеряем bm25_score
            prev = by_id[cid]
            if prev.get('distance') is None and item.get('distance') is not None:
                prev['distance'] = item['distance']
            if prev.get('bm25_score') is None and item.get('bm25_score') is not None:
                prev['bm25_score'] = item['bm25_score']
            # metadata/text предпочтем непустые
            if not prev.get('text') and item.get('text'):
                prev['text'] = item['text']
            if (not prev.get('metadata')) and item.get('metadata'):
                prev['metadata'] = item.get('metadata')

    merged = list(by_id.values())

    cos_sims: List[float] = []
    bm25_scores: List[float] = []
    for it in merged:
        dist = it.get('distance')
        cos = (1.0 - float(dist)) if isinstance(dist, (float, int)) else 0.0
        cos_sims.append(cos)
        bm25_scores.append(float(it.get('bm25_score') or 0.0))

    cos_norm = _minmax_norm(cos_sims)
    bm25_norm = _minmax_norm(bm25_scores)
    a = max(0.0, min(1.0, float(alpha)))
    for i, it in enumerate(merged):
        it['hybrid_score'] = a * cos_norm[i] + (1.0 - a) * bm25_norm[i]

    merged.sort(key=lambda x: float(x.get('hybrid_score') or 0.0), reverse=True)
    return merged[:n_results]


def _fmt_score(x: Optional[float]) -> str:
    return f"{x:.3f}" if isinstance(x, (float, int)) else "n/a"


# Тест retrieval (с query expansion для dense; BM25 ловит редкие токены в исходном вопросе)
test_query = "What are potential targets for Alzheimer's disease treatment?"
expanded = _default_query_expansions(test_query)
test_results = retrieve_chunks(test_query, n_results=5, extra_queries=expanded, use_bm25=True)

print(f"\nТест запрос: {test_query}")
print(f"Extra query (dense only): {expanded[0]}")
print(f"\nНайдено чанков: {len(test_results)}\n")
for i, chunk in enumerate(test_results):
    dist = chunk.get('distance')
    cos_sim = (1.0 - float(dist)) if isinstance(dist, (float, int)) else None
    print(
        f"--- Результат {i+1} "
        f"(hybrid={_fmt_score(chunk.get('hybrid_score'))}, "
        f"bm25={_fmt_score(chunk.get('bm25_score'))}, "
        f"distance={_fmt_score(dist)}, cos_sim~={_fmt_score(cos_sim)}) ---"
    )
    print(f"Title: {chunk.get('metadata', {}).get('title','')[:80]}...")
    print(f"Text: {chunk.get('text','')[:150]}...")
    print()

BM25 index ready: 564 documents

Тест запрос: What are potential targets for Alzheimer's disease treatment?
Extra query (dense only): What are potential targets for Alzheimer's disease treatment? targets BACE1 gamma secretase PSEN1 PSEN2 APP TREM2 NLRP3 APOE4 APOE tau amyloid beta

Найдено чанков: 5

--- Результат 1 (hybrid=0.981, bm25=11.269, distance=0.033, cos_sim~=0.967) ---
Title: Current and Emerging Pharmacological Targets for the Treatment of Alzheimer's Di...
Text: Current and Emerging Pharmacological Targets for the Treatment of Alzheimer's Disease.

No cure or disease-modifying therapy for Alzheimer's disease (...

--- Результат 2 (hybrid=0.960, bm25=10.959, distance=0.051, cos_sim~=0.949) ---
Title: Identification of potential therapeutic targets for Alzheimer's disease from the...
Text: Identification of potential therapeutic targets for Alzheimer's disease from the proteomes of plasma and cerebrospinal fluid in a multicenter Mendelia...

--- Результат 3 (hybrid=0.930, bm2

In [ ]:
"""
## Generation: LLM для ответов (Ollama)
"""

import requests
from typing import Optional, Dict, Any

OLLAMA_URL = "http://127.0.0.1:11434"

def _ollama_url(path: str) -> str:
    return f"{OLLAMA_URL.rstrip('/')}/{path.lstrip('/')}"

def _ollama_is_running(timeout_s: float = 1.0) -> bool:
    try:
        r = requests.get(_ollama_url("/api/tags"), timeout=timeout_s)
        return r.status_code == 200
    except requests.RequestException:
        return False

def call_ollama(
    prompt: str,
    *,
    model: str = "llama3.2:3b",
    timeout_s: float = 60.0,
    temperature: float = 0.1,
    options: Optional[Dict[str, Any]] = None,
) -> str:
    """Вызов локального LLM через Ollama (устойчиво к ошибкам).

    Настройки:
    - temperature по умолчанию низкая, чтобы снизить "болтовню" и нарушения формата.
    - options прокидывается в Ollama /api/generate как есть.
    """
    if not _ollama_is_running():
        return (
            "Ollama API недоступна (localhost:11434).\n\n"
            "Починка (macOS):\n"
            "  1) brew install ollama\n"
            "  2) ollama serve\n"
            f"  3) ollama pull {model}\n\n"
            "Проверка API: curl http://127.0.0.1:11434/api/tags\n"
        )

    generate_url = _ollama_url("/api/generate")
    merged_options: Dict[str, Any] = {
        # conservative defaults (работают почти везде)
        "temperature": float(temperature),
        "top_p": 0.9,
        "repeat_penalty": 1.1,
        "num_predict": 512,
    }
    if options:
        merged_options.update(options)

    try:
        response = requests.post(
            generate_url,
            json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "options": merged_options,
            },
            timeout=timeout_s,
        )

        if response.status_code == 404:
            err_text = response.text.strip()
            try:
                err_json = response.json()
                err_text = err_json.get("error") or err_text
            except ValueError:
                pass

            return (
                "Ollama ответила 404 Not Found.\n"
                f"URL запроса: {generate_url}\n"
                f"Детали: {err_text or '(empty response)'}\n\n"
                "Проверка моделей: ollama list\n"
                "Проверка API: curl http://127.0.0.1:11434/api/tags\n"
            )

        response.raise_for_status()
        data = response.json()
        return data.get("response", "(no 'response' field in JSON)")

    except requests.Timeout:
        return (
            "Timeout при обращении к Ollama.\n"
            "Попробуй увеличить timeout_s или выбрать более лёгкую модель."
        )
    except requests.RequestException as e:
        return f"Ошибка запроса к Ollama API: {e}"
    except ValueError:
        return "Ollama вернула не-JSON ответ. Проверь логи `ollama serve`."

# Быстрый тест
test_response = call_ollama("Say hello in one sentence", temperature=0.1)
print(f"LLM тест: {test_response}")

LLM тест: Hello!


In [53]:
"""
## RAG Pipeline: Retrieval + Generation
"""

# %%
from IPython.display import display, Markdown
from typing import List, Dict, Any, Optional, Set
import re


def _minmax_relevance(distances):
    """Конвертирует distance (меньше = лучше) в relevance (0..1, больше = лучше)."""
    dists = [float(d) for d in distances]
    d_min = min(dists)
    d_max = max(dists)
    denom = d_max - d_min
    if denom <= 1e-12:
        return [1.0 for _ in dists]
    rels = [1.0 - ((d - d_min) / denom) for d in dists]
    return [max(0.0, min(1.0, r)) for r in rels]


def _extract_candidate_targets_from_context(text: str) -> List[str]:
    """Извлекает похожие на таргеты токены из контекста (только из текста)."""
    must_have = {
        "BACE1",
        "TREM2",
        "NLRP3",
        "APOE",
        "APOE4",
        "APP",
        "PSEN1",
        "PSEN2",
        "GSK3B",
        "MAPT",
        "NMDA",
    }
    tokens = set(re.findall(r"\b[A-Z0-9]{3,10}\b", text))
    stop = {"AD", "DNA", "RNA", "CNS", "BBB", "FDA", "PET", "MMSE", "URL", "FAD", "PMID", "ALZFORUM"}
    tokens = {t for t in tokens if t not in stop}
    candidates = sorted(must_have.intersection(tokens))
    extra = sorted(tokens - must_have)[:20]
    return candidates + extra


_CIT_RE = re.compile(r"\[Source\s+(\d+)\]", re.IGNORECASE)

# Drug-like codes should NOT be treated as molecular targets in "molecular_only" mode.
# This is intentionally conservative and mainly blocks common clinical-program codes.
_DRUG_LIKE_RE = re.compile(
    r"\b(?:AZD|JNJ|E|LY|PF|MK|BMS|RO|BI|VX)-?\d{3,9}\b|\b[A-Z]{2,6}-\d{3,9}\b|\b[A-Z]{2,6}\d{3,9}\b" ,
    re.IGNORECASE,
)


def _extract_cited_sources(answer: str) -> Set[int]:
    return {int(m.group(1)) for m in _CIT_RE.finditer(answer or "")}


def _has_citation(line: str) -> bool:
    return bool(_CIT_RE.search(line or ""))


def _strip_citations(text: str) -> str:
    return _CIT_RE.sub("", text or "").strip()


def _normalize_greek(text: str) -> str:
    if not text:
        return ""
    return (
        text.replace("β", "beta")
        .replace("α", "alpha")
        .replace("γ", "gamma")
        .replace("δ", "delta")
    )


# Canonical target -> accepted synonym variants.
# Keep synonyms HIGH-PRECISION: avoid overly broad words that can match unrelated text.
TARGET_SYNONYMS: Dict[str, List[str]] = {
    "tau protein": [
        "tau",
        "microtubule-associated protein tau",
        "mapt",
        "p-tau",
        "phosphorylated tau",
        "hyperphosphorylated tau",
    ],
    "bdnf receptor": [
        "bdnf",
        "trkb",
        "ntrk2",
        "brain-derived neurotrophic factor",
        "tropomyosin receptor kinase b",
    ],
    "bace1": [
        "bace1",
        "bace-1",
        "beta-secretase",
        "β-secretase",
    ],
    "gamma-secretase": [
        "gamma-secretase",
        "γ-secretase",
        "presenilin",
        "psen1",
        "psen2",
    ],
    "amyloid beta": [
        "amyloid beta",
        "amyloid-β",
        "amyloid β",
        "aβ",
        "abeta",
        "beta-amyloid",
        "β-amyloid",
    ],
    "amyloid precursor protein": [
        "amyloid precursor protein",
        "app",
    ],
    "apoe": [
        "apoe",
        "apoe4",
        "apolipoprotein e",
    ],
    "nlrp3": [
        "nlrp3",
        "nlrp3 inflammasome",
    ],
    "trem2": [
        "trem2",
    ],
    "gsk3b": [
        "gsk3b",
        "gsk-3b",
        "gsk3β",
        "gsk3 beta",
        "glycogen synthase kinase 3 beta",
    ],
}


def _norm(s: str) -> str:
    return _normalize_greek((s or "").strip()).lower()


def _find_canonical_key_for_target(target: str) -> Optional[str]:
    """Если таргет похож на один из canonical keys/синонимов — возвращает canonical key."""
    t = _norm(_strip_citations(target))
    if not t:
        return None

    for canonical, syns in TARGET_SYNONYMS.items():
        c = _norm(canonical)
        if c and (c in t or t in c):
            return canonical
        for s in syns:
            ss = _norm(s)
            if ss and (ss in t or t in ss):
                return canonical

    return None


def _synonym_variants_for_target(target: str) -> List[str]:
    """Список строк для сопоставления таргета с контекстом (target + его синонимы)."""
    variants: List[str] = []
    base = _strip_citations(target)
    if base:
        variants.append(base)

    canonical = _find_canonical_key_for_target(base)
    if canonical:
        variants.append(canonical)
        variants.extend(TARGET_SYNONYMS.get(canonical, []))

    # Дедуп + нормализация пробелов
    out: List[str] = []
    seen: Set[str] = set()
    for v in variants:
        vv = " ".join((v or "").split())
        key = _norm(vv)
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(vv)
    return out


def _parse_targets_section(answer: str) -> List[str]:
    if not answer:
        return []

    lines = [ln.rstrip("\n") for ln in answer.splitlines()]
    start = None
    for i, ln in enumerate(lines):
        if ln.strip().lower() == "targets:":
            start = i + 1
            break
    if start is None:
        return []

    targets: List[str] = []
    for ln in lines[start:]:
        s = ln.strip()
        if not s:
            continue
        if s.lower().startswith("rationale:"):
            break
        if s.startswith("-") or s.startswith("•"):
            item = s.lstrip("-•").strip()
            item = _strip_citations(item)
            if "—" in item:
                target = item.split("—", 1)[0].strip()
            elif "-" in item:
                target = item.split("-", 1)[0].strip()
            else:
                target = item.strip()
            if target:
                targets.append(target)
    return targets


def _is_molecular_target(target: str) -> bool:
    t = (target or "").strip()
    if not t:
        return False

    # Reject drug-development codes (AZD3293, JNJ-54861911, etc.)
    if _DRUG_LIKE_RE.search(t):
        return False

    # Filter obvious non-target acronyms that can leak from metadata/context.
    if t.upper() in {"URL", "PMID", "FAD", "ALZFORUM", "BACE", "AZD"}:
        return False

    tl = _norm(t)

    banned_substrings = [
        "amyloid plaque",
        "amyloid plaques",
        "plaques",
        "plaque",
        "neurofibrillary",
        "tangles",
        "tangle",
        "hallmark",
        "peripheral protein",
        "peripheral proteins",
        "biomarker",
        "biomarkers",
        "symptom",
        "symptoms",
        "pathology",
        "glial cells",
        "cell type",
    ]
    if any(b in tl for b in banned_substrings):
        return False

    allow_phrases = [
        "amyloid precursor protein",
        "app",
        "tau",
        "amyloid-beta",
        "amyloid beta",
        "abeta",
        "beta-amyloid",
        "presenilin",
        "gamma-secretase",
        "secretase",
        "gsk3",
        "gsk-3",
        "bace1",
        "beta-secretase",
    ]
    if any(a in tl for a in allow_phrases):
        return True

    if re.search(r"\b[A-Z0-9]{2,10}\b", t):
        return True

    if any(k in tl for k in ["receptor", "kinase", "inflammasome", "interleukin", "tnf", "pathway", "complement"]):
        return True

    return False


def _extract_key_tokens(target: str) -> List[str]:
    if not target:
        return []
    t = _normalize_greek(target)
    raw = re.findall(r"[A-Za-z0-9]{2,}", t)
    stop = {
        "protein",
        "proteins",
        "gene",
        "genes",
        "receptor",
        "receptors",
        "enzyme",
        "subunit",
        "pathway",
        "complex",
        "signaling",
        "modulation",
    }
    toks = [x.lower() for x in raw if x.lower() not in stop]
    return toks[:5]


def _target_supported_by_context(target: str, context: str) -> bool:
    if not target:
        return False

    ctx = _norm(context or "")

    # 1) direct / synonym match
    for v in _synonym_variants_for_target(target):
        vv = _norm(v)
        if not vv:
            continue
        # если это фраза/с дефисом — substring, иначе word-boundary
        if any(ch in vv for ch in [" ", "-", "/"]):
            if vv in ctx:
                return True
        else:
            if re.search(rf"\b{re.escape(vv)}\b", ctx):
                return True

    # 2) fallback token match
    t_clean = _norm(_strip_citations(target))
    if t_clean and t_clean in ctx:
        return True
    for tok in _extract_key_tokens(target):
        if re.search(rf"\b{re.escape(tok)}\b", ctx):
            return True

    return False


def _extract_allowed_targets_from_context(context: str) -> List[str]:
    ctx_raw = _normalize_greek(context or "")
    ctx = ctx_raw.lower()
    allowed: Set[str] = set()

    # 1) Gene-like tokens
    for t in re.findall(r"\b[A-Z0-9]{3,10}\b", ctx_raw):
        # filter out years / PMIDs / numeric-only tokens
        if t.isdigit():
            continue
        if re.fullmatch(r"19\d{2}|20\d{2}|2100", t):
            continue
        if t not in {"AD", "DNA", "RNA", "CNS", "BBB", "FDA", "PET", "MMSE", "URL", "FAD", "PMID", "ALZFORUM", "LOAD", "MCI"}:
            allowed.add(t)

    # 2) Canonicals if any synonym is present
    for canonical, syns in TARGET_SYNONYMS.items():
        present = False
        for s in [canonical] + list(syns):
            ss = _norm(s)
            if not ss:
                continue
            if any(ch in ss for ch in [" ", "-", "/"]):
                if ss in ctx:
                    present = True
                    break
            else:
                if re.search(rf"\b{re.escape(ss)}\b", ctx):
                    present = True
                    break
        if present:
            allowed.add(canonical)

    # 3) A few precise lower markers (kept for backwards compatibility)
    lower_markers = [
        "tau",
        "abeta",
        "beta-amyloid",
        "amyloid-beta",
        "amyloid precursor protein",
        "gamma-secretase",
        "secretase",
        "presenilin",
        "nmda",
        "trem2",
        "nlrp3",
        "apoe",
        "complement",
    ]
    for m in lower_markers:
        if m in ctx:
            allowed.add(m)

    return sorted(allowed)

In [56]:
"""
## RAG Pipeline (continued): Prompt + Validation + Orchestration
"""

from typing import List, Dict, Any, Optional, Tuple
import time
import re


def _build_sources_and_context(chunks: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], str]:
    """Builds `sources` metadata and a text `context` for the prompt."""
    sources: List[Dict[str, Any]] = []
    ctx_lines: List[str] = []
    for i, c in enumerate(chunks, start=1):
        meta = c.get("metadata") or {}
        title = (meta.get("title") or c.get("title") or "").strip()
        url = (meta.get("url") or c.get("url") or "").strip()
        year = str(meta.get("year") or c.get("year") or "")
        text = (c.get("text") or "").strip()
        rel = c.get("relevance")
        if rel is None:
            dist = c.get("distance")
            if isinstance(dist, (float, int)):
                rel = max(0.0, min(1.0, 1.0 - float(dist)))
            else:
                rel = 0.0

        sources.append(
            {
                "title": title,
                "url": url,
                "year": year,
                "text": text,
                "relevance": float(rel) if isinstance(rel, (float, int)) else 0.0,
            }
        )

        ctx_lines.append(f"[Source {i}] {title} ({year})")
        if url:
            ctx_lines.append(f"URL: {url}")
        ctx_lines.append(text)
        ctx_lines.append("")

    return sources, "\n".join(ctx_lines).strip()


def _build_prompt(
    question: str,
    context: str,
    *,
    strict: bool,
    molecular_only: bool,
    allowed_targets: List[str],
) -> str:
    allowed_preview = ", ".join(allowed_targets[:25])
    rules = [
        "You are a biomedical research assistant.",
        "Answer using ONLY the provided Sources.",
        "Output MUST follow the exact format below; no preamble, no extra sections.",
        "",
        "FORMAT:",
        "Targets:",
        "- <target> — <evidence sentence> [Source N]",
        "Rationale:",
        "- <1-2 sentences explaining why these are plausible targets> [Source N]",
    ]
    if strict:
        rules.extend(
            [
                "",
                "STRICT RULES:",
                "- Every bullet MUST contain at least one citation like [Source 1].",
                "- Targets and rationale must be BULLET LINES only.",
                "- Do NOT cite sources you did not receive.",
                "- Do NOT introduce targets that are not supported by the Sources.",
            ]
        )
    if molecular_only:
        rules.extend(
            [
                "",
                "MOLECULAR-ONLY MODE:",
                "- Targets must be molecular targets (genes/proteins/receptors/enzymes/pathways).",
                "- Do NOT output drug codes or clinical compound identifiers as targets.",
            ]
        )
    if allowed_targets:
        rules.extend(["", "ALLOWED TARGET HINTS (not exhaustive):", allowed_preview])

    return (
        "\n".join(rules)
        + "\n\nQUESTION:\n"
        + question.strip()
        + "\n\nSOURCES:\n"
        + context.strip()
    )


def _split_answer_sections(answer: str) -> Tuple[Optional[List[str]], Optional[List[str]], List[str]]:
    """Returns (targets_lines, rationale_lines, structural_errors)."""
    lines = [ln.rstrip("\n") for ln in (answer or "").splitlines()]

    t_idx = [i for i, ln in enumerate(lines) if ln.strip().lower() == "targets:"]
    r_idx = [i for i, ln in enumerate(lines) if ln.strip().lower() == "rationale:"]

    errors: List[str] = []
    if len(t_idx) != 1:
        errors.append("Targets: header must appear exactly once")
        return None, None, errors
    if len(r_idx) != 1:
        errors.append("Rationale: header must appear exactly once")
        return None, None, errors

    t0 = t_idx[0]
    r0 = r_idx[0]
    if r0 <= t0:
        errors.append("Rationale: must come after Targets:")
        return None, None, errors

    targets_block = lines[t0 + 1 : r0]
    rationale_block = lines[r0 + 1 :]
    return targets_block, rationale_block, errors


def _parse_targets_from_block(targets_block: List[str]) -> List[str]:
    targets: List[str] = []
    for ln in targets_block:
        s = ln.strip()
        if not s:
            continue
        if not (s.startswith("-") or s.startswith("•")):
            continue
        item = s.lstrip("-•").strip()
        item = _strip_citations(item)
        if "—" in item:
            target = item.split("—", 1)[0].strip()
        elif "-" in item:
            target = item.split("-", 1)[0].strip()
        else:
            target = item.strip()
        if target:
            targets.append(target)
    return targets


def validate_answer(
    answer: str,
    *,
    context: str,
    n_sources: int,
    strict: bool,
    molecular_only: bool,
    allowed_targets: Optional[List[str]] = None,
    require_target_supported: bool = True,
    max_targets: int = 10,
) -> Dict[str, Any]:
    """Validates format + citations + grounding. Returns a report dict."""
    errors: List[str] = []
    answer = (answer or "").strip()
    if not answer:
        return {"ok": False, "errors": ["Empty answer"], "targets": []}

    targets_block, rationale_block, structural_errors = _split_answer_sections(answer)
    errors.extend(structural_errors)
    if targets_block is None or rationale_block is None:
        return {"ok": False, "errors": errors, "targets": []}

    # Targets: only bullet lines
    targets_bullets = [ln.strip() for ln in targets_block if ln.strip()]
    if not targets_bullets:
        errors.append("Targets: section is empty")
    for ln in targets_bullets:
        if not (ln.startswith("-") or ln.startswith("•")):
            errors.append("Targets: contains non-bullet content")
            break
        if strict and not _has_citation(ln):
            errors.append("Targets: bullet without citation")
            break
        if "—" not in ln:
            errors.append("Targets: bullet must contain an em dash '—'")
            break

    # Rationale: only bullet lines
    rationale_bullets = [ln.strip() for ln in rationale_block if ln.strip()]
    if not rationale_bullets:
        errors.append("Rationale: section is empty")
    for ln in rationale_bullets:
        if not (ln.startswith("-") or ln.startswith("•")):
            errors.append("Rationale: contains non-bullet content")
            break
        if strict and not _has_citation(ln):
            errors.append("Rationale: bullet without citation")
            break

    targets = _parse_targets_from_block(targets_block)
    if not targets:
        errors.append("No targets parsed from Targets: bullets")
    elif len(targets) > max_targets:
        errors.append(f"Too many targets parsed: {len(targets)} > {max_targets}")

    # Duplicate targets (case-insensitive)
    norm_targets = [_norm(t) for t in targets]
    dups = sorted({t for t in norm_targets if norm_targets.count(t) > 1})
    if dups:
        errors.append(f"Duplicate targets: {', '.join(dups[:10])}")

    cited = _extract_cited_sources(answer)
    bad = sorted([i for i in cited if i < 1 or i > int(n_sources)])
    if bad:
        errors.append(f"Cites out-of-range sources: {bad}")

    if molecular_only and targets:
        non_mol = [t for t in targets if not _is_molecular_target(t)]
        if non_mol:
            errors.append(f"Non-molecular targets in molecular-only mode: {', '.join(non_mol[:5])}")

    if require_target_supported and targets:
        unsupported = [t for t in targets if not _target_supported_by_context(t, context)]
        if unsupported:
            errors.append(f"Targets not supported by context: {', '.join(unsupported[:5])}")

    if allowed_targets:
        allowed_norm = {_norm(a) for a in allowed_targets}
        outside = []
        for t in targets:
            tn = _norm(t)
            if tn in allowed_norm:
                continue
            if _find_canonical_key_for_target(t) is not None:
                continue
            outside.append(t)
        if strict and outside:
            errors.append(f"Targets not in allowed set (strict): {', '.join(outside[:5])}")

    ok = len(errors) == 0
    return {
        "ok": ok,
        "errors": errors,
        "targets": targets,
        "n_targets": len(targets),
        "cited_sources": sorted(list(cited)),
    }


def _fmt_val_report(v: Dict[str, Any]) -> str:
    ok = bool(v.get("ok"))
    errs = v.get("errors") or []
    targets = v.get("targets") or []
    cited = v.get("cited_sources") or []
    lines = [f"- OK: {'OK' if ok else 'FAIL'}"]
    if targets:
        lines.append(f"- Targets parsed: {len(targets)}")
    if cited:
        lines.append(f"- Cited sources: {', '.join(map(str, cited))}")
    if errs:
        lines.append("- Errors:")
        for e in errs[:10]:
            lines.append(f"  - {e}")
    return "\n".join(lines)


def _find_source_for_target(target: str, sources: List[Dict[str, Any]]) -> int:
    variants = _synonym_variants_for_target(target)
    for i, s in enumerate(sources, start=1):
        txt = _norm(s.get("text", ""))
        for v in variants:
            vv = _norm(v)
            if not vv:
                continue
            if any(ch in vv for ch in [" ", "-", "/"]):
                if vv in txt:
                    return i
            else:
                if re.search(rf"\b{re.escape(vv)}\b", txt):
                    return i
    return 1


def _salvage_answer(
    *,
    context: str,
    sources: List[Dict[str, Any]],
    molecular_only: bool,
    max_targets: int = 5,
) -> str:
    # Prefer a small prioritized set of plausible AD molecular targets to avoid
    # leaking metadata tokens (e.g., URL/FAD) into Targets:.
    priority = ["BACE1", "APP", "MAPT", "TREM2", "NLRP3", "APOE", "PSEN1", "PSEN2", "GSK3B", "TNF"]
    chosen: List[str] = []
    for t in priority:
        if len(chosen) >= int(max_targets):
            break
        if molecular_only and (not _is_molecular_target(t)):
            continue
        if _target_supported_by_context(t, context):
            chosen.append(t)

    # Fill remaining slots from extracted allowed targets (already filtered upstream).
    allowed = _extract_allowed_targets_from_context(context)
    if molecular_only:
        allowed = [t for t in allowed if _is_molecular_target(t)]
    allowed = [t for t in allowed if _target_supported_by_context(t, context)]
    for t in allowed:
        if len(chosen) >= int(max_targets):
            break
        if t not in chosen:
            chosen.append(t)

    lines: List[str] = ["Targets:"]
    for t in chosen:
        sid = _find_source_for_target(t, sources)
        lines.append(f"- {t} — Mentioned in the retrieved context. [Source {sid}]")

    # If still empty, keep strict format but expect validation to mark it as bad.
    if not chosen:
        lines.append("- APP — No supported targets could be extracted; defaulting to a common AD target. [Source 1]")

    lines.append("Rationale:")
    lines.append("- Evidence is limited to the retrieved abstracts/snippets; the answer may be incomplete. [Source 1]")
    return "\n".join(lines)


def rag_query(
    question: str,
    *,
    n_chunks: int = 5,
    strict: bool = True,
    molecular_only: bool = True,
    max_retries: int = 2,
    temperature: float = 0.0,
    model: str = "llama3.2:3b",
    use_bm25: bool = True,
) -> Dict[str, Any]:
    t0 = time.time()

    extra = _default_query_expansions(question)
    chunks = retrieve_chunks(
        question,
        n_results=int(n_chunks),
        extra_queries=extra,
        use_bm25=bool(use_bm25),
    )
    sources, context = _build_sources_and_context(chunks)
    allowed_targets = _extract_allowed_targets_from_context(context)

    prompt = _build_prompt(
        question,
        context,
        strict=bool(strict),
        molecular_only=bool(molecular_only),
        allowed_targets=allowed_targets,
    )

    last_validation: Dict[str, Any] = {}
    used_salvage = False
    answer: str = ""
    for attempt in range(int(max_retries) + 1):
        answer = call_ollama(prompt, model=model, temperature=float(temperature))
        last_validation = validate_answer(
            answer,
            context=context,
            n_sources=len(sources),
            strict=bool(strict),
            molecular_only=bool(molecular_only),
            allowed_targets=allowed_targets,
        )
        if last_validation.get("ok"):
            break
        prompt = (
            prompt
            + "\n\nREPAIR INSTRUCTIONS:\n"
            + "- Output ONLY the required sections; no extra text.\n"
            + "- Targets/Rationale must be bullet lines and each bullet must cite [Source N].\n"
            + "- Use only targets supported by SOURCES.\n"
            + ("- Molecular-only: do NOT output drug codes as targets.\n" if molecular_only else "")
        )

    if not last_validation.get("ok") and strict:
        used_salvage = True
        answer = _salvage_answer(
            context=context,
            sources=sources,
            molecular_only=bool(molecular_only),
        )
        last_validation = validate_answer(
            answer,
            context=context,
            n_sources=len(sources),
            strict=bool(strict),
            molecular_only=bool(molecular_only),
            allowed_targets=allowed_targets,
        )

    dt = time.time() - t0
    return {
        "question": question,
        "answer": answer,
        "sources": [{k: v for k, v in s.items() if k != "text"} for s in sources],
        "context": context,
        "validation": last_validation,
        "used_salvage": bool(used_salvage),
        "elapsed_s": dt,
    }


In [ ]:
"""
## Evaluation: Метрики качества
"""

test_questions = [
    "What are potential targets for Alzheimer's disease treatment?",
    "Are the targets druggable with small molecules, biologics, or other modalities?",
    "What is the role of tau protein in Alzheimer's?",
    "What are promising therapeutic approaches for neuroinflammation in AD?",
]

print("Оценка retrieval качества (min-max relevance на top-5):\n")

retrieval_scores = []
for q in test_questions:
    chunks = retrieve_chunks(q, n_results=5)
    if not chunks:
        print(f"Q: {q[:60]}...")
        print("   No chunks returned")
        print()
        continue

    distances = [c.get('distance', 0.0) for c in chunks]
    relevances = _minmax_relevance(distances)
    avg_relevance = sum(relevances) / len(relevances)

    retrieval_scores.append(avg_relevance)

    print(f"Q: {q[:60]}...")
    print(f"   Avg relevance: {avg_relevance:.3f}")
    print(f"   Top chunk: {chunks[0]['metadata']['title'][:60]}...")
    print()

if retrieval_scores:
    print(f"Средний retrieval score: {sum(retrieval_scores)/len(retrieval_scores):.3f}")


Оценка retrieval качества (min-max relevance на top-5):

Q: What are potential targets for Alzheimer's disease treatment...
   Avg relevance: 0.469
   Top chunk: Current and Emerging Pharmacological Targets for the Treatme...

Q: Are the targets druggable with small molecules, biologics, o...
   Avg relevance: 0.359
   Top chunk: Current and Emerging Pharmacological Targets for the Treatme...

Q: What is the role of tau protein in Alzheimer's?...
   Avg relevance: 0.473
   Top chunk: Role of tau protein in Alzheimer's disease: The prime pathol...

Q: What are promising therapeutic approaches for neuroinflammat...
   Avg relevance: 0.463
   Top chunk: Drug delivery strategies with lipid-based nanoparticles for ...

Средний retrieval score: 0.441


In [ ]:
print("Оценка generation качества )")

# Важно: делаем параметры максимально стабильными и близкими к demo-ячейке
EVAL_N_CHUNKS = 5
EVAL_MAX_RETRIES = 2
EVAL_TEMPERATURE = 0.0

for q in test_questions[:2]:  # первые 2 вопроса
    result = rag_query(
        q,
        n_chunks=EVAL_N_CHUNKS,
        strict=True,
        molecular_only=True,
        max_retries=EVAL_MAX_RETRIES,
        temperature=EVAL_TEMPERATURE,
        use_bm25=True,
    )

    print(f"\nQ: {q}")
    print(_fmt_val_report(result.get("validation", {})))

    # Короткий head для быстрой диагностики формата
    ans = result.get("answer", "") or ""
    print(f"A (head): {ans[:260]}...")
    print()


ОЦЕНКА GENERATION

Q: What are potential targets for Alzheimer's disease treatment?
Validation: OK
A (head): Targets:
- Amyloid precursor protein — believed to be involved in the development of Alzheimer's disease [Source 2]

Rationale:
- I can only list targets explicitly mentioned in the retrieved context....


Q: Are the targets druggable with small molecules, biologics, or other modalities?
Validation: OK
A (head): Targets:
- BACE1 — Small-molecule BACE1 inhibitors have been extensively developed for the last 20 years [Source 1]
- gamma-secretase — γ-Secretase is a membrane embedded aspartyl protease complex with presenilin as the catalytic component [Source 4]
- secreta...



In [ ]:
"""
## Evaluation

Оценка качества RAG системы на 3 метриках:
- Citation Accuracy (цитирует ли LLM источники)
- Target Coverage (находит ли известные мишени)
- Retrieval Quality (качество semantic search)
"""
import numpy as np

def quick_eval():
    """Быстрая оценка RAG"""
    
    # Тесты
    questions = [
        "What are potential targets for Alzheimer's disease treatment?",
        "What is the role of tau protein in Alzheimer's?",
        "Are BACE1 inhibitors effective?",
        "How does amyloid-beta contribute to AD?",
        "What is the role of neuroinflammation?",
    ]
    
    targets = ['tau protein', 'amyloid-beta', 'BACE1', 'APOE', 'TREM2']
    
    # 1. Citation
    citations = []
    for q in questions:
        ans = rag_query(q, n_chunks=3, strict=False, molecular_only=False)['answer']
        n_citations = ans.count('[Source')
        n_sentences = len([s for s in ans.split('.') if s.strip()])
        rate = min(n_citations / max(n_sentences, 1), 1.0)  # cap at 1.0
        citations.append(rate)    
    # 2. Coverage
    found = sum(1 for t in targets 
                if any(t.lower() in c['text'].lower() 
                      for c in retrieve_chunks(f"role of {t}", 5)))
    
    # 3. Retrieval
    retrieval = [np.mean([1 - c['distance'] 
                         for c in retrieve_chunks(q, 5)]) 
                for q in questions]
    
    # Results
    print("="*50)
    print("EVALUATION RESULTS")
    print("="*50)
    print(f"Citation Accuracy: {np.mean(citations):.2f}")
    print(f"Target Coverage:   {found}/{len(targets)} ({found/len(targets):.0%})")
    print(f"Retrieval Quality: {np.mean(retrieval):.3f}")
    print("="*50)
    
    return {
        'citation': np.mean(citations),
        'coverage': found/len(targets),
        'retrieval': np.mean(retrieval)
    }



In [ ]:

eval_results = quick_eval()

# Пример ответа
print("\nExample:")
res = rag_query("What are potential targets for Alzheimer's?", n_chunks=3, strict=False)
print(f"Q: What are potential targets for Alzheimer's?")
print(f"A: {res['answer'][:200]}...")
print(f"Citations: {'OK' if '[Source' in res['answer'] else 'FAIL'}")

EVALUATION RESULTS
Citation Accuracy: 0.80
Target Coverage:   4/5 (80%)
Retrieval Quality: 0.940

Example:
Q: What are potential targets for Alzheimer's?
A: Targets:
- Amyloid precursor protein — believed to be involved in the development of Alzheimer's disease [Source 1]
- Presenilin 1 — a key component of the gamma-secretase complex, which is implicated...
Citations: OK


In [ ]:
"""
## Интерактивный интерфейс
"""

# %%
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

# Виджеты
query_input = widgets.Textarea(
    value='What are potential targets for Alzheimer\'s disease treatment?',
    placeholder='Enter your question...',
    description='Question:',
    layout=widgets.Layout(width='100%', height='80px')
)

n_chunks_slider = widgets.IntSlider(
    value=5,
    min=3,
    max=10,
    step=1,
    description='Chunks:',
    continuous_update=False
)

molecular_only_toggle = widgets.Checkbox(
    value=True,
    description='Molecular targets only',
)

search_button = widgets.Button(
    description='Search',
    button_style='primary',
    icon='search'
)

output_area = widgets.Output()

# Стабильные настройки генерации для UI (меньше "NOT OK" из-за формата)
UI_MAX_RETRIES = 2
UI_TEMPERATURE = 0.0


def on_search_click(b):
    """Обработчик поиска"""
    output_area.clear_output()

    with output_area:
        display(HTML("<h3>Searching...</h3>"))

        # RAG query
        result = rag_query(
            query_input.value,
            n_chunks=n_chunks_slider.value,
            strict=True,
            molecular_only=bool(molecular_only_toggle.value),
            max_retries=UI_MAX_RETRIES,
            temperature=UI_TEMPERATURE,
            use_bm25=True,
        )

        output_area.clear_output()

        # Validation
        display(HTML("<h3>Validation:</h3>"))
        display(Markdown(_fmt_val_report(result.get('validation', {}))))

        # Отображение ответа
        display(HTML("<h3>Answer:</h3>"))
        display(Markdown(result['answer']))

        # Отображение источников
        display(HTML("<h3>Sources:</h3>"))
        for i, source in enumerate(result['sources']):
            rel = source.get('relevance')
            rel_val = float(rel) if isinstance(rel, (float, int)) else 0.0
            relevance_bar = "█" * int(rel_val * 10)
            display(HTML(f"""
            <div style='margin: 10px 0; padding: 10px; background: #f5f5f5; border-left: 3px solid #4CAF50;'>
                <b>{i+1}. {source['title']}</b><br>
                <small>Year: {source['year']} | Relevance: {relevance_bar} {rel_val:.2f}</small><br>
                <a href='{source['url']}' target='_blank'>View on PubMed →</a>
            </div>
            """))


search_button.on_click(on_search_click)

# Отображение интерфейса
display(HTML("<h2>Alzheimer's Drug Target Research Assistant</h2>"))
display(query_input)
display(n_chunks_slider)
display(molecular_only_toggle)
display(search_button)
display(output_area)

Textarea(value="What are potential targets for Alzheimer's disease treatment?", description='Question:', layou…

IntSlider(value=5, continuous_update=False, description='Chunks:', max=10, min=3)

Checkbox(value=True, description='Molecular targets only')

Button(button_style='primary', description='Search', icon='search', style=ButtonStyle())

Output()

## Часть 4. Расширение модальностей и выбор моделей

### Исходное состояние решения
В текущем прототипе используется одна основная модальность данных — неструктурированный текст научных публикаций (PubMed: заголовок + аннотация; порядка сотни статей). Далее выполняются очистка текста, разбиение на чанки, индексация (dense embeddings + векторное хранилище), гибридный поиск (dense + BM25) и генерация ответа локальной LLM. Для повышения воспроизводимости и снижения галлюцинаций применяются строгие требования к формату ответа, пост-валидация с повторными попытками и проверка того, что перечисленные молекулярные мишени действительно присутствуют в извлечённом контексте (с учётом синонимов).

---

### 1) На какие модальности данных можно расширить решение?

#### 1.1. Структурированные биомедицинские знания (приоритет для drug discovery)
Данный класс источников целесообразно подключать первым, поскольку он дополняет текст статей нормализованными фактами и идентификаторами.

- **Белок/ген (UniProt, STRING, Human Protein Atlas)**: функция, локализация, экспрессия в релевантных тканях/клетках, сети взаимодействий; полезно для биологического обоснования и контекста (например, мозг/микроглия).
- **Лекарства/химические соединения (ChEMBL, DrugBank, PubChem)**: связь «мишень → молекулы/модальность», количественные параметры активности (IC50/EC50), сведения о селективности и профиле; критично для вопросов о druggability.
- **Клинические исследования (ClinicalTrials.gov и аналоги)**: связь «препарат/механизм → фаза/статус/результаты»; позволяет оценивать зрелость направления и снижать риск выбора “перспективной” мишени без клинической поддержки.

#### 1.2. Биологические пути и функциональные аннотации
- **Pathways и онтологии (Reactome/KEGG/GO)**: связь «мишень → путь → downstream эффекты», функциональные термины и модули; повышает качество причинно-следственных объяснений и поддерживает системный взгляд.

#### 1.3. Омиксные данные (для валидации релевантности мишени в заболевании)
Подключение омиксных данных оправдано, если требуется не только перечисление целей из литературы, но и подтверждение изменений в AD.

- **Транскриптомика (GTEx, Allen Brain Atlas, single-cell атласы)**: ткане- и клеточно-специфическая экспрессия, дифференциальная экспрессия в AD/контроль.
- **Протеомика и PTM (PRIDE и др.)**: изменения на уровне белка и модификаций (важно для случаев, где ключевую роль играет PTM).


---

### 2) Как это можно сделать?

#### 2.1. Стратегия интеграции для структурированных источников (рекомендуемая стартовая траектория)
1. **Извлечение и нормализация сущностей (ETL)**: приведение к устойчивым идентификаторам (gene/protein/drug IDs), дедупликация и контроль качества.
2. **Представление в виде “fact cards”**: компактные, проверяемые утверждения (включая численные значения и ссылки на первоисточник), пригодные для включения в контекст.
3. **Индексация и поиск**: хранение фактов как отдельного корпуса (с метаданными “модальность/источник/ID сущности”) и совместное использование с корпусом статей.

#### 2.2. Мультиретривер и слияние результатов
- Для текста: гибридный поиск dense + BM25.
- Для структурных данных: API/SQL-запросы (или прединдексированные факты).
- Далее: **fusion** (веса/правила/переранжирование) и формирование единого контекста для генерации.

#### 2.3. Маршрутизация запросов (query routing)
Для повышения точности полезно выбирать источники в зависимости от типа вопроса:
- druggability/активность/препараты → Drug DB;
- экспрессия/клеточные типы → HPA/Allen/GTEx;
- “на какой стадии клиники” → Clinical trials;
- механизм/путь → Reactome/KEGG/GO.

#### 2.4. Knowledge Graph 
При росте сложности запросов целесообразно добавить граф сущностей (мишени–препараты–пути–публикации–испытания). Это повышает объяснимость (можно показывать цепочку связей), но требует отдельного контура извлечения отношений и поддержки актуальности графа.

---

### 3) Какие модели и почему выбраны для решения?

#### 3.1. Модель эмбеддингов для биомедицинского текста
Используется **`pritamdeka/S-PubMedBert-MS-MARCO`** как модель для dense retrieval.

Обоснование выбора:
- модель ориентирована на поиск по научному/биомедицинскому тексту и устойчивее к доменной лексике и аббревиатурам;
- в условиях ограниченного корпуса (аннотации PubMed) является рациональным компромиссом между качеством и вычислительными затратами.

#### 3.2. Гибридный ретривал (BM25 + dense)
Гибрид применяется для снижения риска пропуска редких ключевых токенов (например, имен генов/белков) и повышения полноты извлечения контекста. При этом требуется контроль, чтобы “ключевые совпадения” не вытесняли семантически более релевантные источники; именно поэтому используется не чистый BM25, а комбинация.

#### 3.3. Генеративная модель (LLM)
Используется локальная LLM через Ollama: **`llama3.2:3b`**.

Обоснование выбора:
- соответствует ограничениям прототипа (локально, без платных API, приватность данных, работоспособность на Mac);
- качество ответа стабилизируется инженерными мерами: строгий формат, валидация, повторные попытки и фильтрация неподдержанных мишеней.

#### 3.4. Векторное хранилище
Используется **ChromaDB** как простое локальное хранилище, достаточное для прототипирования и малых объёмов. При росте корпуса и потребности в фильтрации/масштабировании разумной заменой являются Qdrant или Weaviate.


## На какие вопросы система отвечает лучше/хуже

### Наблюдаемая область применимости

Важно различать два уровня качества:
1) **Формальная корректность** (прошла ли валидацию: формат + ссылки + отсутствие неподдержанных “мишеней”).
2) **Содержательная полезность** (насколько ответ действительно отвечает на поставленный вопрос и не подменяет тип сущностей).

**Система отвечает наиболее устойчиво на вопросы, решаемые экстрактивно из извлечённого контекста** (PubMed title+abstract), когда требуется:
- перечислить **молекулярные мишени**, *явно упомянутые* в retrieved context;
- кратко описать роль/механизм конкретной мишени *в пределах найденных аннотаций*;
- выделить, какие мишени фигурируют в обзорах по терапии/нейровоспалению/амилоидному каскаду;
- поддержать каждый пункт ссылками на источники.

**Система отвечает ограниченно, если вопрос требует данных, которые редко присутствуют в аннотациях или отсутствуют в текущем корпусе**, например:
- количественные параметры (IC50/EC50), детальная фармакология и сравнение конкретных соединений;
- клинический статус/исходы по конкретным препаратам без подключения регистров клинических испытаний;
- полнота “списка всех X” (например, все FDA‑approved препараты на дату), если этого нет в retrieved context;
- генерация гипотез “вне источников” (в строгом режиме такие утверждения подавляются).

Ключевые причины:
- корпус ограничен аннотациями (не full‑text), поэтому часть фактов/чисел отсутствует;
- строгая валидация снижает риск галлюцинаций, но может приводить к коротким ответам (0–3 пункта);
- LLM небольшого размера (`llama3.2:3b`) иногда нарушает формат, что компенсируется ретраями.

Ниже приведён прогон набора вопросов. Его следует интерпретировать так:
- `ok=True` означает только то, что ответ *соответствует правилам* (формат + citations + grounding).
- `used_salvage=True` означает, что сработал fallback: система сгенерировала минимальный grounded ответ, чтобы соблюсти формат/ссылки (это повышает формальную “надёжность”, но обычно снижает содержательную полноту).
- `limited=True` — эвристический флаг: модель явно указала ограниченность данных/контекста; это часто ожидаемо в строгом режиме и не всегда означает «плохой» ответ.
- если вопрос **drug‑centric**, а включён режим *Molecular targets only*, то drug‑код/идентификатор соединения в `Targets:` считается ошибкой (несоответствие типа сущностей) и должен приводить к `ok=False`; для таких вопросов корректнее отключать molecular‑only или вводить отдельную схему ответа (например, `Drugs:` отдельно от `Targets:`).

In [57]:

import time
import re
import pandas as pd


def _validation_reasons(v: dict, max_items: int = 4) -> str:
    """Compactly describes why validation failed.

    Supports both the older rich validation dict (multiple fields)
    and the newer `errors: [...]` shape.
    """
    if not v:
        return "(no validation)"
    if v.get("ok"):
        return "OK"

    # New shape
    errs = v.get("errors")
    if errs:
        if isinstance(errs, list):
            if len(errs) > max_items:
                return f"errors={errs[:max_items]} (+{len(errs)-max_items})"
            return f"errors={errs}"
        return f"errors={errs}"

    # Legacy shape
    parts = []
    for k in [
        "invalid_citations",
        "missing_sections",
        "bad_target_format_lines",
        "missing_citation_lines",
        "hallucinated_targets",
        "non_molecular_targets",
        "duplicate_targets",
    ]:
        val = v.get(k)
        if not val:
            continue
        if isinstance(val, list):
            if len(val) > max_items:
                parts.append(f"{k}={val[:max_items]} (+{len(val)-max_items})")
            else:
                parts.append(f"{k}={val}")
        else:
            parts.append(f"{k}={val}")

    return "; ".join(parts) if parts else "NOT_OK"


# Грубые эвристики для интерпретации: что модель выдала в качестве "Targets"
_DRUG_CODE_RE = re.compile(
    r"\b(?:AZD\d{3,6}|JNJ-?\d{5,9}|E\d{3,5}|LY\d{3,6}|PF-?\d{3,6}|MK-?\d{3,6}|BMS-?\d{3,6}|RO\d{3,6}|BI\d{3,6}|VX-?\d{3,6})\b",
    re.IGNORECASE,
)


def _is_drug_like(s: str) -> bool:
    t = (s or "").strip()
    if not t:
        return False
    if _DRUG_CODE_RE.search(t):
        return True
    # общая форма "AB1234" / "ABC-12345"
    if re.search(r"\b[A-Z]{2,6}-?\d{3,7}\b", t.upper()):
        return True
    return False


def _is_gene_symbol_like(s: str) -> bool:
    t = (s or "").strip()
    if not t:
        return False
    if _is_drug_like(t):
        return False
    return bool(re.fullmatch(r"[A-Z0-9]{2,10}", t))


def _is_molecular_phrase_like(s: str) -> bool:
    tl = (s or "").strip().lower()
    if not tl:
        return False
    return any(
        kw in tl
        for kw in [
            "protein",
            "receptor",
            "kinase",
            "secretase",
            "inflammasome",
            "cytokine",
            "interleukin",
            "pathway",
            "complement",
        ]
    )


def _question_is_drug_centric(q: str) -> bool:
    ql = (q or "").lower()
    return any(
        kw in ql
        for kw in [
            "ic50",
            "ec50",
            "inhibitor",
            "inhibitors",
            "drug",
            "drugs",
            "fda-approved",
            "approved",
            "small molecule",
            "biologic",
            "antibody",
        ]
    )


def _behavior_label(*, ok: bool, limited: bool, drug_like_targets: int, drug_centric_question: bool) -> str:
    if not ok:
        return "NOT_OK (format/grounding)"
    if drug_centric_question and drug_like_targets == 0:
        return "OK, но ответ не по сущностям (скорее target-centric)"
    if drug_like_targets > 0 and not drug_centric_question:
        return "OK, но подмена сущностей (drug-like в Targets)"
    if limited:
        return "OK, но ограничено контекстом"
    return "OK"


QUESTION_SPECS = [
    {
        "class": "Target-centric (экстрактивный список)",
        "q": "List molecular targets for Alzheimer's disease that are explicitly mentioned in the retrieved context.",
        "expectation": "Ожидается grounded ответ; число таргетов зависит от top-k контекста.",
    },
    {
        "class": "Target-centric (роль конкретной мишени)",
        "q": "What is the role of tau (MAPT) in Alzheimer's disease according to the retrieved papers?",
        "expectation": "Ожидается grounded ответ, если tau/MAPT присутствует в retrieved context.",
    },
    {
        "class": "Target-centric (нейровоспаление)",
        "q": "Which molecular targets related to neuroinflammation are mentioned (e.g., TREM2, NLRP3, complement)?",
        "expectation": "Ожидается grounded ответ при наличии соответствующих обзорных источников в top-k.",
    },
    {
        "class": "Mixed (druggability без внешних БД)",
        "q": "Are any of the mentioned targets druggable with small molecules or biologics? Answer only from the retrieved context.",
        "expectation": "Может быть grounded, но детали (числа/перечни препаратов) часто отсутствуют в abstracts.",
    },
    {
        "class": "Drug-centric (IC50/соединения)",
        "q": "Provide IC50 values for the best BACE1 inhibitors and compare them.",
        "expectation": "В текущем target-centric формате часто некорректно по смыслу или резко ограничено контекстом.",
    },
    {
        "class": "Drug-centric (регуляторный/список)",
        "q": "List all FDA-approved Alzheimer's drugs as of 2025 and their mechanisms.",
        "expectation": "В строгом режиме будет ограничено retrieved context; полный список без внешней БД недостижим.",
    },
    {
        "class": "Out-of-scope (генерация гипотез вне источников)",
        "q": "Propose 10 novel Alzheimer's targets not mentioned in the sources and design new inhibitors.",
        "expectation": "Строгий режим должен подавлять неподдержанные гипотезы и уходить в ограниченный grounded ответ.",
    },
]

EVAL = {
    "n_chunks": 8,
    "max_retries": 2,
    "temperature": 0.0,
    "strict": True,
    "molecular_only": True,
    "use_bm25": True,
}

rows = []
for spec in QUESTION_SPECS:
    q = spec["q"]
    t0 = time.time()
    res = rag_query(q, **EVAL)
    dt = time.time() - t0

    used_salvage = bool(res.get("used_salvage"))

    v = res.get("validation", {})
    ans = res.get("answer", "") or ""
    targets = _parse_targets_section(ans)

    # индикаторы "система ограничилась контекстом"
    ans_l = ans.lower()
    limitation_flag = any(
        ph in ans_l
        for ph in [
            "evidence is limited",
            "limited to",
            "may be incomplete",
            "only from the retrieved",
            "only the provided",
        ]
    )

    drug_like = [t for t in targets if _is_drug_like(t)]
    gene_like = [t for t in targets if _is_gene_symbol_like(t)]
    mol_phrase = [t for t in targets if _is_molecular_phrase_like(t)]

    drug_centric_q = _question_is_drug_centric(q)

    ok = bool(v.get("ok"))
    behavior = _behavior_label(
        ok=ok,
        limited=bool(limitation_flag),
        drug_like_targets=len(drug_like),
        drug_centric_question=bool(drug_centric_q),
    )

    answer_head = "\n".join((ans.splitlines() or [])[:8])

    rows.append(
        {
            "class": spec["class"],
            "question": q,
            "ok": ok,
            "behavior": behavior,
            "n_targets": len(targets),
            "targets_head": ", ".join(targets[:4]),
            "drug_like_targets": ", ".join(drug_like[:3]),
            "gene_like_targets": ", ".join(gene_like[:3]),
            "used_salvage": used_salvage,
            "limited": bool(limitation_flag),
            "reasons": _validation_reasons(v),
            "sec": round(dt, 1),
            "top_source": (res.get("sources") or [{}])[0].get("title", "")[:80],
            "answer_head": answer_head,
        }
    )


df_eval = pd.DataFrame(rows)

# Показываем компактную таблицу (без длинных строк ответа)
display(
    df_eval[[
        "class",
        "ok",
        "behavior",
        "n_targets",
        "targets_head",
        "drug_like_targets",
        "used_salvage",
        "limited",
        "sec",
        "top_source",
    ]]
)

print("\nСводка по классам вопросов:")
if not df_eval.empty:
    summary = (
        df_eval.groupby("class")
        .agg(
            n=("question", "count"),
            ok_rate=("ok", "mean"),
            avg_targets=("n_targets", "mean"),
            limited_rate=("limited", "mean"),
            avg_sec=("sec", "mean"),
        )
        .sort_values("ok_rate", ascending=False)
    )
    display(summary)

print("\nПримеры (первые строки ответов):")
for i, r in enumerate(rows[:3], start=1):
    print(f"\n[{i}] {r['class']}")
    print(r["question"])
    print(r["answer_head"])


,class,ok,behavior,n_targets,targets_head,drug_like_targets,used_salvage,limited,sec,top_source
0,Target-centric (экстрактивный список),True,"OK, но ограничено контекстом",5,"BACE1, APP, MAPT, PSEN1",,True,True,38.6,Amyloid β-based therapy for Alzheimer's diseas...
1,Target-centric (роль конкретной мишени),True,"OK, но ограничено контекстом",5,"BACE1, MAPT, abeta, amyloid beta",,True,True,45.2,Tau-targeting antisense oligonucleotide MAPTRx...
2,Target-centric (нейровоспаление),True,"OK, но ограничено контекстом",5,"BACE1, APP, MAPT, TNF",,True,True,34.0,Tau and neuroinflammation in Alzheimer's disea...
3,Mixed (druggability без внешних БД),True,"OK, но ответ не по сущностям (скорее target-ce...",4,"BACE1, γ-Secretase, Presenilin, APP",,False,False,34.4,BACE1 Inhibitors for Alzheimer's Disease: Curr...
4,Drug-centric (IC50/соединения),True,"OK, но ответ не по сущностям (скорее target-ce...",5,"BACE1, APP, ASBI, BACE2",,True,True,37.6,BACE1 Inhibitors for Alzheimer's Disease: Curr...
5,Drug-centric (регуляторный/список),True,"OK, но ответ не по сущностям (скорее target-ce...",5,"BACE1, APP, MAPT, PSEN1",,True,True,37.3,Anti-amyloid-β Antibodies and Anti-tau Therapi...
6,Out-of-scope (генерация гипотез вне источников),True,"OK, но ответ не по сущностям (скорее target-ce...",5,"BACE1, APP, MAPT, PSEN1",,True,True,66.7,BACE-1 Inhibitors Targeting Alzheimer's Disease.



Сводка по классам вопросов:


,n,ok_rate,avg_targets,limited_rate,avg_sec
class,,,,,
Drug-centric (IC50/соединения),1,1.0,5.0,1.0,37.6
Drug-centric (регуляторный/список),1,1.0,5.0,1.0,37.3
Mixed (druggability без внешних БД),1,1.0,4.0,0.0,34.4
Out-of-scope (генерация гипотез вне источников),1,1.0,5.0,1.0,66.7
Target-centric (нейровоспаление),1,1.0,5.0,1.0,34.0
Target-centric (роль конкретной мишени),1,1.0,5.0,1.0,45.2
Target-centric (экстрактивный список),1,1.0,5.0,1.0,38.6



Примеры (первые строки ответов):

[1] Target-centric (экстрактивный список)
List molecular targets for Alzheimer's disease that are explicitly mentioned in the retrieved context.
Targets:
- BACE1 — Mentioned in the retrieved context. [Source 6]
- APP — Mentioned in the retrieved context. [Source 4]
- MAPT — Mentioned in the retrieved context. [Source 8]
- PSEN1 — Mentioned in the retrieved context. [Source 4]
- PSEN2 — Mentioned in the retrieved context. [Source 4]
Rationale:
- Evidence is limited to the retrieved abstracts/snippets; the answer may be incomplete. [Source 1]

[2] Target-centric (роль конкретной мишени)
What is the role of tau (MAPT) in Alzheimer's disease according to the retrieved papers?
Targets:
- BACE1 — Mentioned in the retrieved context. [Source 5]
- MAPT — Mentioned in the retrieved context. [Source 1]
- abeta — Mentioned in the retrieved context. [Source 2]
- amyloid beta — Mentioned in the retrieved context. [Source 2]
- amyloid-beta — Mentioned in the retriev